# Klassifikation mit allen Werten

`JobSat` als Zielwert


In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix


## Daten laden

Im One-Hot-Encoded Datensatz sind die One-Hot-Encoded Spalten mit bool Werten. Um es etwas einfacher zu gestalten werden diese Werte hier in Integer-Werte (False -> 0; True -> 1) umgewandelt.

In [4]:

df = pd.read_csv("survey_results_cleaned_final.csv")


bool_cols = df.select_dtypes(include=['bool']).columns

for col in bool_cols:
    df[col] = df[col].astype(int)
df.dtypes

ResponseId                          int64
MainBranch                         object
Age                                object
AgeNum                            float64
EdLevel                            object
Employment                         object
WorkExp                           float64
LearnCodeAI                        object
YearsCode                         float64
DevType                            object
OrgSize                            object
ICorPM                             object
RemoteWork                         object
RemoteCategoryNum                 float64
Industry                           object
AIThreat                           object
NewRole                            object
Country                            object
LanguageChoice                     object
LanguageHaveWorkedWith             object
DatabaseChoice                     object
DatabaseHaveWorkedWith             object
PlatformChoice                     object
PlatformHaveWorkedWith            

## Zielvariable in Klassen einteilen 

Dadurch haben wir mehr Trainingsdaten, als wenn wir `JobSat` von 0-10 als Klassen defonieren würden

Klassen
- **Low**: 0–3
- **Medium**: 4–6
- **High**: 7–10


In [5]:
# Nur Zeilen behalten, wo JobSat vorhanden ist
df = df.dropna(subset=["JobSat"]).copy()

def map_jobsat(x):
    x = float(x)
    if x <= 3:
        return "Low"
    elif x <= 6:
        return "Medium"
    else:
        return "High"


## Feature-Spalten bestimmen

- Textspalten: `object` (Strings)
- Numerische Spalten: `int/float`



In [6]:
# Textspalten (Strings)
text_cols = df.select_dtypes(include=["object"]).columns.tolist()

df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

# Numerische Spalten
num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

# Zielspalte aus numerischen Features entfernen (falls vorhanden)
num_cols = [c for c in num_cols if c != "JobSat"] #?

df = df.dropna(subset=["__text__"] + num_cols + ["JobSat"])

y = df["JobSat"].apply(map_jobsat)

print("Textspalten:", text_cols)
print("Numerische Spalten:", num_cols)

# Feature-Matrix aus den ausgewählten Spalten
X = df[["__text__"] + num_cols].copy()


X.head()

Textspalten: ['MainBranch', 'Age', 'EdLevel', 'Employment', 'LearnCodeAI', 'DevType', 'OrgSize', 'ICorPM', 'RemoteWork', 'Industry', 'AIThreat', 'NewRole', 'Country', 'LanguageChoice', 'LanguageHaveWorkedWith', 'DatabaseChoice', 'DatabaseHaveWorkedWith', 'PlatformChoice', 'PlatformHaveWorkedWith', 'WebframeChoice', 'WebframeHaveWorkedWith', 'DevEnvsChoice', 'DevEnvsHaveWorkedWith', 'OfficeStackAsyncHaveWorkedWith', 'CommPlatformHaveWorkedWith', 'AIModelsChoice', 'AIModelsHaveWorkedWith', 'AISelect', 'AIAgents', 'AIAgent_Uses']
Numerische Spalten: ['ResponseId', 'AgeNum', 'WorkExp', 'YearsCode', 'RemoteCategoryNum', 'RemoteMissing', 'ConvertedCompTotal']


,__text__,ResponseId,AgeNum,WorkExp,YearsCode,RemoteCategoryNum,RemoteMissing,ConvertedCompTotal
0,i am a developer by profession 25-34 years old...,1,29.0,8.0,14.0,0.00,0,61659.84
1,i am a developer by profession 25-34 years old...,2,29.0,2.0,10.0,0.25,0,105102.00
3,i am a developer by profession 35-44 years old...,4,39.0,4.0,5.0,0.00,0,36435.36
4,i am a developer by profession 35-44 years old...,5,39.0,21.0,22.0,0.50,1,60000.00
5,i am a developer by profession 45-54 years old...,6,49.0,15.0,20.0,0.50,1,120000.00


## Train/Test Split



In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train class distribution:\n", y_train.value_counts(normalize=True))
print("Test class distribution:\n", y_test.value_counts(normalize=True))

Train size: 10258
Test size: 2565
Train class distribution:
 JobSat
High      0.718464
Medium    0.220121
Low       0.061415
Name: proportion, dtype: float64
Test class distribution:
 JobSat
High      0.718519
Medium    0.220273
Low       0.061209
Name: proportion, dtype: float64


## Preprocessing für Text und numerische Spalten




In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(), "__text__"),
        ("num", StandardScaler(), num_cols)
    ],
    remainder="drop"
)


## Pipeline definieren

- Preprocessing
- Feature-Selektion (SelectFromModel mit L1-LinearSVC)
- Klassifikator (LinearSVC)


In [11]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(LinearSVC(penalty="l1", dual=False, C=0.5))),
    ("classifier", LinearSVC())
])



## GridSearchCV



In [12]:

parameters = {
    # TF-IDF: nur word, nur die zwei wichtigsten Varianten
    "preprocessing__text__analyzer": ["word"],
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__max_df": [0.9],
    "preprocessing__text__min_df": [2],

    # Klassifikator: 2 sinnvolle Regularisierungen + optional balancing
    "classifier__C": [1.0, 2.0],
    "classifier__class_weight": [None, "balanced"],
}

grid = GridSearchCV(pipeline, param_grid=parameters, verbose=2, cv=3, n_jobs=-1)


## Grid Search + Beste Parameter


In [13]:

grid.fit(X_train, y_train)

print("Beste Performance:", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


C:\Users\MoritzSchwarz\PycharmProjects\data-analytics-project\.venv\Lib\site-packages\sklearn\svm\_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Beste Performance: 0.7165138684350212
Beste Parameter:
 {'classifier__C': 1.0, 'classifier__class_weight': None, 'preprocessing__text__analyzer': 'word', 'preprocessing__text__max_df': 0.9, 'preprocessing__text__min_df': 2, 'preprocessing__text__ngram_range': (1, 1)}


## Evaluation auf Testdaten


In [14]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Classification Report (Test):")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=["Low", "Medium", "High"]))

Classification Report (Test):
              precision    recall  f1-score   support

        High       0.74      0.97      0.84      1843
         Low       0.00      0.00      0.00       157
      Medium       0.45      0.14      0.21       565

    accuracy                           0.72      2565
   macro avg       0.40      0.37      0.35      2565
weighted avg       0.63      0.72      0.65      2565

Confusion Matrix (rows=true, cols=pred):
[[   0   33  124]
 [   0   77  488]
 [   2   60 1781]]


In [15]:
y_pred_train = best_model.predict(X_train)

print("Classification Report (Train):")
print(classification_report(y_train, y_pred_train))

Classification Report (Train):
              precision    recall  f1-score   support

        High       0.75      0.97      0.85      7370
         Low       0.72      0.03      0.05       630
      Medium       0.47      0.14      0.22      2258

    accuracy                           0.73     10258
   macro avg       0.65      0.38      0.37     10258
weighted avg       0.69      0.73      0.66     10258

